In [1]:
# Run the for loop in each function to return a dictionary
# Make a seperate function that turns each dictionary into a database 

In [2]:
from datetime import datetime
from dotenv import load_dotenv
import pandas as pd
import requests
import sqlite3
import os
import yfinance as yf

In [3]:
load_dotenv()

True

In [4]:
fmg_api_key = os.getenv("FMG_API_KEY")
tiingo_key = os.getenv("TIINGO_TOKEN")

In [5]:
symbols_string = "AAPL, TSLA, AMZN, MSFT, NVDA, GOOGL, META, NFLX, JPM, V, BAC, PYPL, DIS, T, PFE, COST, INTC, KO, TGT, NKE, SPY, BA, BABA, XOM, WMT, GE, CSCO, VZ, JNJ, CVX, PLTR, SQ, SHOP, SBUX, SOFI, HOOD, RBLX, SNAP, AMD, UBER, FDX, ABBV, ETSY, MRNA, LMT, GM, F, LCID, CCL, DAL, UAL, AAL, TSM, SONY, ET, MRO, COIN, RIVN, RIOT, CPRX, VWO, SPYG, NOK, ROKU, VIAC, ATVI, BIDU, DOCU, ZM, PINS, TLRY, WBA, MGM, NIO, C, GS, WFC, ADBE, PEP, UNH, CARR, HCA, TWTR, BILI, SIRI, FUBO, RKT"
symbols_split = symbols_string.split(',')
symbols = [s.strip() for s in symbols_split]

### Get Financial Statements

In [6]:
def get_income_statement(ticker, k):
    # This function will return the income statement, balance sheet, and cash flow statement of a company
    income_endpoint = f"https://financialmodelingprep.com/stable/income-statement?symbol={ticker}&apikey={k}"
    income_data = requests.get(income_endpoint)
    income = income_data.json()
    return income

def get_balance_sheet(ticker, k):
    balance_endpoint = f"https://financialmodelingprep.com/stable/balance-sheet-statement?symbol={ticker}&apikey={k}"
    balance_data = requests.get(balance_endpoint)
    balance = balance_data.json()
    return balance

def get_cash_flow_statement(ticker, k):
    cash_endpoint = f"https://financialmodelingprep.com/stable/cash-flow-statement?symbol={ticker}&apikey={k}"
    cash_data = requests.get(cash_endpoint)
    cash = cash_data.json()
    return cash

### Price History

In [7]:
def yfinance_historic_price(ticker):
    yfinance_ticker = yf.Ticker(ticker)
    price_history = yfinance_ticker.history('5y')
    price_history = price_history.reset_index()

    return price_history

### Share History

In [8]:
def yfinance_get_shares(ticker):
    yfinance_ticker = yf.Ticker(ticker)
    shares = yfinance_ticker.get_shares_full(start='2021-10-01')
    df_shares = pd.DataFrame(data=shares,  index=None,)
    df_shares.reset_index(inplace=True)
    df_shares = df_shares.rename(columns={'index': 'Date', 0: 'Shares'})
    df_shares['Date'] = pd.to_datetime(df_shares['Date']).dt.date

    return df_shares

In [9]:
def get_fiscal_years(ticker, key):
    income_statement = get_income_statement(ticker=ticker, k=key)

    fiscal_years = []
    for year in income_statement:
        fiscal_year = year['fiscalYear']
        fiscal_years.append(fiscal_year)

    return fiscal_years

In [22]:
fiscal_years = get_fiscal_years(ticker='AAPL', key=fmg_api_key)
fiscal_years_dict = {'Fiscal Year': fiscal_years}

In [10]:
test_stocks = ['AAPL', 'TSLA', 'AMZN']

### P/B ratio function

In [16]:
pb_ratios_list = []
def calc_pb_ratio(ticker, balance):
    # Call api to obtain company balance sheet
    result = []
    for year in balance:
        # Find filing date
        file_date = year['filingDate']
        
        # Find price on filing date
        price_history = yfinance_historic_price(ticker=ticker)
        filing_date_stock_price = price_history[price_history['Date'] == file_date]
        
        # Convert file date to datetime for next step
        file_date = pd.to_datetime(file_date)
        # Find closest date on shares df
        shares_history = yfinance_get_shares(ticker=ticker)
        shares_history['Date'] = pd.to_datetime(shares_history['Date'])
        closest_date = (shares_history['Date'] - file_date).abs().idxmin()
        closest_row = shares_history.loc[closest_date]
        # Calculate Market Cap
        market_cap = closest_row['Shares'] * filing_date_stock_price['Close'].iloc[0]
        market_cap = float(market_cap)

        # Calculate pb ratio
        pb_ratio = round(market_cap / year['totalStockholdersEquity'], 2)
        result.append(pb_ratio)
    
    return result

for stock in test_stocks:
    balance_sheet = get_balance_sheet(ticker=stock, k=fmg_api_key)
    pb_ratio = calc_pb_ratio(ticker=stock, balance=balance_sheet)
    pb_ratios_list.append(pb_ratio)
    
pb_ratios_list


[[54.26, 60.01, 43.88, 48.0, 38.03],
 [19.03, 17.62, 9.71, 12.26, 10.35],
 [5.49, 8.43, 8.84, 7.25, 0.58]]

In [18]:
pb_ratios_dict = dict(zip(test_stocks, pb_ratios_list))
pb_ratios_dict

{'AAPL': [54.26, 60.01, 43.88, 48.0, 38.03],
 'TSLA': [19.03, 17.62, 9.71, 12.26, 10.35],
 'AMZN': [5.49, 8.43, 8.84, 7.25, 0.58]}

In [23]:
pb_ratios_full_dict = fiscal_years_dict | pb_ratios_dict
pb_ratios_full_dict

{'Fiscal Year': ['2025', '2024', '2023', '2022', '2021'],
 'AAPL': [54.26, 60.01, 43.88, 48.0, 38.03],
 'TSLA': [19.03, 17.62, 9.71, 12.26, 10.35],
 'AMZN': [5.49, 8.43, 8.84, 7.25, 0.58]}

In [24]:
df_pb_ratios = pd.DataFrame.from_dict(data=pb_ratios_full_dict)
df_pb_ratios

,Fiscal Year,AAPL,TSLA,AMZN
0,2025,54.26,19.03,5.49
1,2024,60.01,17.62,8.43
2,2023,43.88,9.71,8.84
3,2022,48.00,12.26,7.25
4,2021,38.03,10.35,0.58


### Debt to Equity function

In [24]:
def calc_de_ratio(ticker, key):
    for year in balance:
        # Step 1: Pull shareholder and total debt from balance sheet
        shareholder_equity = year['totalStockholdersEquity']
        total_debt = year['totalDebt']
        # Step 2: Divide numbers
        de_ratio = total_debt / shareholder_equity
        # Step 3: Append yearly ratios to list
        de_ratios.append(de_ratio)

### Create Dataframe from various list of P/B ratios

In [11]:
def create_pb_df(ticker, key):
    pb_ratios, x = calc_pb_ratio(ticker, key)
    pb_df = pd.DataFrame({'Fiscal_Years': fiscal_years, 'P/B_ratio': pb_ratios})

    return pb_df

test = create_pb_df(ticker='AAPL', key=api_key)
test

ValueError: All arrays must be of the same length

In [9]:

# Debt to equity
de_dict = {}
# Revenue Growth
revenue_growth_dict = {}
# Gross Profit margin
gross_margin_dict = {}
# Altman z score
z_score = {}

In [31]:
pb_keys = ['Fiscal Years']
pb_values = []
keys_to_delete = []

In [28]:
fiscal_years = get_fiscal_years(ticker="AAPL", key=api_key)
pb_values.append(fiscal_years)

In [30]:
for s in symbols:
    # Step 1: Obtain financial statements
    income_statement = get_income_statement(ticker=s,)
    balance_sheet = get_balance_sheet(ticker=s,)
    cash_flow_stat = get_cash_flow_statement(ticker=s,)
    # Step 2: Calculate P/B ratios
    pb_ratios, skipped = calc_pb_ratio(ticker=s, key=api_key), bala
    pb_values.append(pb_ratios)
    keys_to_delete.append(skipped)

$SQ: possibly delisted; no price data found  (period=5y) (Yahoo error = "No data found, symbol may be delisted")
$SQ: possibly delisted; no price data found  (period=5y) (Yahoo error = "No data found, symbol may be delisted")
$SQ: possibly delisted; no price data found  (period=5y) (Yahoo error = "No data found, symbol may be delisted")
$SQ: possibly delisted; no price data found  (period=5y) (Yahoo error = "No data found, symbol may be delisted")
$SQ: possibly delisted; no price data found  (period=5y) (Yahoo error = "No data found, symbol may be delisted")
$MRO: possibly delisted; no price data found  (period=5y) (Yahoo error = "No data found, symbol may be delisted")
$MRO: possibly delisted; no price data found  (period=5y) (Yahoo error = "No data found, symbol may be delisted")
$MRO: possibly delisted; no price data found  (period=5y) (Yahoo error = "No data found, symbol may be delisted")
$MRO: possibly delisted; no price data found  (period=5y) (Yahoo error = "No data found, symb

In [ ]:
pb_ratios

In [ ]:
# PB ratio
pb_ratio_dict = dict(zip(pb_keys